<a href="https://colab.research.google.com/github/databyhuseyn/DeepLearning/blob/main/Fine_Tuning_with_Unsloth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 3.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.2/924

In [1]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments, Trainer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [44]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-2-7b-bnb-4bit",
    max_seq_length = 2048
)

==((====))==  Unsloth 2026.5.10: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-2-7b-bnb-4bit as a legacy tokenizer.


In [45]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'v_proj'],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias='none',
    use_gradient_checkpointing = True,
    random_state=42
)

In [46]:
import pandas as pd

In [47]:
df = pd.read_parquet("hf://datasets/iamtarun/code_instructions_120k_alpaca/data/train-00000-of-00001-d9b93805488c263e.parquet")


In [48]:
df

,instruction,input,output,prompt
0,Create a function to calculate the sum of a se...,"[1, 2, 3, 4, 5]",# Python code\ndef sum_sequence(sequence):\n ...,Below is an instruction that describes a task....
1,Develop a function that will add two strings,"str1 = ""Hello ""\nstr2 = ""world""","def add_strings(str1, str2):\n """"""This func...",Below is an instruction that describes a task....
2,Design a data structure in C++ to store inform...,,#include <map>\n#include <string>\n\nclass Gro...,Below is an instruction that describes a task....
3,Implement a sorting algorithm to sort a given ...,"[3, 1, 4, 5, 9, 0]",def bubble_sort(arr):\n n = len(arr)\n \n ...,Below is an instruction that describes a task....
4,Design a Swift application for tracking expens...,Not applicable,import UIKit\n\nclass ExpenseViewController: U...,Below is an instruction that describes a task....
...,...,...,...,...
121954,Create a JavaScript program that takes in a st...,"""Hello World!""",function mostFrequentLetter(str) {\n // Create...,Below is an instruction that describes a task....
121955,Generate a program in Java that takes a single...,,public class Absolute {\n public static voi...,Below is an instruction that describes a task....
121956,Create a class in C# to store customer details.,,public class Customers\n{\n private string ...,Below is an instruction that describes a task....
121957,Create a classification algorithm in Python to...,,# Import Python libraries\nimport numpy as np\...,Below is an instruction that describes a task....


In [49]:
df.prompt.loc[1]

'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nDevelop a function that will add two strings\n\n### Input:\nstr1 = "Hello "\nstr2 = "world"\n\n### Response:\ndef add_strings(str1, str2):\n    """This function takes two strings and returns the sum of them."""\n    return str1 + str2\n\nstr1 = "Hello "\nstr2 = "world"\nsum_of_strings = add_strings(str1, str2)'

In [50]:
alpaca_prompt = """
Below is an instruction that describes a task, paired with an input provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Output:
{}
"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompt_func(examples):
  instruction = examples['instruction']
  input = examples['input']
  output = examples['output']

  texts = []
  for instruction, input, output in zip(instruction, input, output):
    text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
    texts.append(text)
  return {"text": texts}

In [51]:
dataset = load_dataset('iamtarun/code_instructions_120k_alpaca', split='train')
dataset = dataset.map(formatting_prompt_func, batched=True)
dataset

Dataset({
    features: ['instruction', 'input', 'output', 'prompt', 'text'],
    num_rows: 121959
})

In [53]:
from trl import SFTTrainer
import torch

max_seq_length = 2048


training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=3,
    # num_training_epochs=1,
    learning_rate=2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = 'adamw_8bit',
    weight_decay = 0.01,
    lr_scheduler_type = 'linear',
    seed=42,
    output_dir = 'outputs'
)


trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing=False,
    args = training_args
)

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/121959 [00:00<?, ? examples/s]

In [54]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 121,959 | Num Epochs = 1 | Total steps = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,388,608 of 6,746,804,224 (0.12% trained)


Step,Training Loss
1,1.331836
2,1.436035
3,1.078423


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-3/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-3.


In [56]:
# Streaming

FastLanguageModel.for_inference(model)
inputs = tokenizer([
    alpaca_prompt.format(
        'How to create view in PostgreSQL?',
        '',
        ''
    )
], return_tensors='pt').to('cuda')

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens=2048)

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


```sql
CREATE VIEW view_name AS
SELECT column_name, column_name, column_name
FROM table_name
```
</s>


# Saving the model

In [60]:
from dotenv import load_dotenv
import os

In [59]:
load_dotenv()

True

In [63]:
hf_token = os.getenv('HF_TOKEN')

In [64]:
model.push_to_hub('mrhuseyn4/lora_model_fine_tuned_with_alpaca120', token=hf_token)

README.md:   0%|          | 0.00/545 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|1         |  571kB / 33.6MB            

Saved model to https://huggingface.co/mrhuseyn4/lora_model_fine_tuned_with_alpaca120
